# 06. 차별화 피처 엔지니어링

## 추가 피처 목록

### A. 구매 시점 피처 (Timing Features)
| 피처 | 설명 |
|------|------|
| `up_avg_purchase_interval` | 이 유저가 이 상품을 평균 며칠마다 구매했는가 (prior 기준) |
| `up_std_purchase_interval` | 구매 간격의 표준편차 |
| `up_purchase_regularity` | 구매 규칙성 (1 - CV), 높을수록 규칙적 |
| `up_interval_count` | 구매 간격을 계산할 수 있는 횟수 (구매 횟수 - 1) |
| `days_until_expected` | 평균 간격 - 현재 경과일 (음수 = 이미 살 시점 지남) |
| `timing_ratio` | 현재 경과일 / 평균 간격 (1.0 초과 = 구매 주기 초과) |
| `is_overdue` | timing_ratio >= 1.0 이면 1 |

### B. 공동 구매 패턴 피처 (Co-purchase Features)
| 피처 | 설명 |
|------|------|
| `copurchase_ratio` | 이 상품의 top-K 공동구매 파트너 중 이 유저가 구매한 비율 |
| `copurchase_weighted_score` | 공동구매 빈도로 가중한 친화도 점수 |
| `copurchase_match_count` | 공동구매 파트너 중 실제 구매한 개수 |

### 출력
- `data/prep/k-pick_total_v4.csv` (v3 + 새 피처 10개)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')

# 절대 경로를 1순위로 시도, 실패하면 상대 경로로 fallback
_candidates = [
    Path(r'c:\Users\gksal\capstone-kpick\K-Pick'),  # 절대 경로 (1순위)
    Path.cwd(),
    Path.cwd().parent,
]
BASE = next((p for p in _candidates if (p / 'data' / 'prep').exists()), None)
if BASE is None:
    raise FileNotFoundError(
        f'data/prep 폴더를 찾을 수 없습니다.\n현재 경로: {Path.cwd()}'
    )

RAW  = BASE / 'data' / 'raw'
PREP = BASE / 'data' / 'prep'
print(f'BASE : {BASE}')
print(f'RAW  : {RAW}')
print(f'PREP : {PREP}')
print('라이브러리 로드 완료')

## 1. 원본 데이터 로드

In [ ]:
print('데이터 로드 중...')
orders = pd.read_csv(
    RAW / 'orders.csv',
    usecols=['order_id','user_id','order_number','days_since_prior_order','eval_set']
)
prior_products = pd.read_csv(
    RAW / 'order_products__prior.csv',
    usecols=['order_id','product_id']
)
# 병합 기준 키 + train 주문 경과일만 추출
target = pd.read_csv(
    PREP / 'k-pick_total_v3.csv',
    usecols=['user_id','product_id','days_since_prior_order']
)

# prior 주문만 분리
prior_orders = orders[orders['eval_set'] == 'prior'][
    ['order_id','user_id','order_number','days_since_prior_order']
].copy()
prior_orders['days_since_prior_order'] = prior_orders['days_since_prior_order'].fillna(0)

print(f'prior 주문 수  : {len(prior_orders):,}')
print(f'prior 구매 기록: {len(prior_products):,}')
print(f'target 쌍      : {len(target):,}')

---
## 섹션 A. 구매 시점 피처 (Timing Features)

> **아이디어**: `days_since_prior_order`는 유저의 마지막 주문 이후 경과일이다.  
> 이 값이 `이 상품의 평균 구매 주기`와 얼마나 일치하는지 수치화하면,  
> "지금이 이 상품을 살 타이밍인가"를 피처로 표현할 수 있다.

In [ ]:
# 유저별 주문을 시간순으로 정렬 후 누적 경과일 계산
# → Instacart는 절대 날짜가 없으므로, 첫 주문을 day=0으로 두고 누적합 사용
prior_orders_sorted = prior_orders.sort_values(['user_id','order_number'])
prior_orders_sorted['cum_days'] = (
    prior_orders_sorted.groupby('user_id')['days_since_prior_order'].cumsum()
)

print('누적 날짜 계산 완료')
print(prior_orders_sorted[
    ['user_id','order_number','days_since_prior_order','cum_days']
].head(8).to_string(index=False))

In [ ]:
# prior 구매 기록 + 누적 날짜 병합
pp_timing = prior_products.merge(
    prior_orders_sorted[['order_id','user_id','order_number','cum_days']],
    on='order_id', how='inner'
).sort_values(['user_id','product_id','order_number'])

# 같은 (user, product) 내에서 직전 구매 날짜
pp_timing['prev_cum_days'] = (
    pp_timing.groupby(['user_id','product_id'])['cum_days'].shift(1)
)
# 구매 간격 = 이번 구매 날짜 - 직전 구매 날짜 (첫 구매는 NaN)
pp_timing['purchase_interval'] = pp_timing['cum_days'] - pp_timing['prev_cum_days']

# (user, product) 단위로 집계
timing_agg = (
    pp_timing.dropna(subset=['purchase_interval'])
    .groupby(['user_id','product_id'])['purchase_interval']
    .agg(
        up_avg_purchase_interval='mean',
        up_std_purchase_interval='std',
        up_interval_count='count'
    )
    .reset_index()
)
timing_agg['up_std_purchase_interval'] = timing_agg['up_std_purchase_interval'].fillna(0)

# 규칙성: CV(변동계수)가 낮을수록 규칙적으로 구매 → 재구매 예측 신뢰도 높음
timing_agg['up_purchase_regularity'] = (
    1 - timing_agg['up_std_purchase_interval'] / (timing_agg['up_avg_purchase_interval'] + 1e-6)
).clip(0, 1).astype(np.float32)
timing_agg['up_interval_count'] = timing_agg['up_interval_count'].astype(np.int16)

print(f'timing_agg 행 수: {len(timing_agg):,}')
timing_agg.head()

In [ ]:
# 한 번만 산 상품은 interval 정보 없음 → 유저 평균으로 대체(fallback)
user_avg_fallback = (
    pp_timing.dropna(subset=['purchase_interval'])
    .groupby('user_id')['purchase_interval']
    .mean()
    .reset_index()
    .rename(columns={'purchase_interval': '_fallback'})
)

timing_merged = (
    target[['user_id','product_id','days_since_prior_order']]
    .merge(timing_agg, on=['user_id','product_id'], how='left')
    .merge(user_avg_fallback, on='user_id', how='left')
)
timing_merged['up_avg_purchase_interval'] = (
    timing_merged['up_avg_purchase_interval'].fillna(timing_merged['_fallback'])
)
timing_merged.drop(columns=['_fallback'], inplace=True)

# ── 핵심 파생 피처 ──
# 음수: 이미 살 시점이 지남 (재구매 가능성 ↑)
# 양수: 아직 살 때가 아님
timing_merged['days_until_expected'] = (
    timing_merged['up_avg_purchase_interval'] - timing_merged['days_since_prior_order']
).astype(np.float32)

# 1.0 초과: 평균 구매 주기를 넘김 → 재구매 촉진 신호
timing_merged['timing_ratio'] = (
    timing_merged['days_since_prior_order'] / (timing_merged['up_avg_purchase_interval'] + 1e-6)
).astype(np.float32)

timing_merged['is_overdue'] = (timing_merged['timing_ratio'] >= 1.0).astype(np.int8)

TIMING_COLS = [
    'user_id','product_id',
    'up_avg_purchase_interval','up_std_purchase_interval',
    'up_purchase_regularity','up_interval_count',
    'days_until_expected','timing_ratio','is_overdue'
]
timing_final = timing_merged[TIMING_COLS].copy()

print(f'timing_final 행 수: {len(timing_final):,}')
print(f'is_overdue=1 비율: {timing_final["is_overdue"].mean():.3f}')
timing_final.describe().T.round(3)

---
## 섹션 B. 공동 구매 패턴 피처 (Co-purchase Features)

> **아이디어**: 우유를 자주 사는 유저가 시리얼 코너에서도 구매할 확률이 높다.  
> 각 상품의 "자주 같이 사는 상품 top-K"를 추출하고,  
> 대상 유저가 그 파트너들을 얼마나 구매했는지를 피처로 만든다.
>
> **계산 방식**: `order × product` sparse 행렬 → `A^T @ A` = 상품별 공동구매 횟수
>
> **주의**: 배치 방식으로 계산하므로 수 분 소요될 수 있음

In [ ]:
# target에 등장하는 상품만 대상 (계산량 절감)
target_products = target['product_id'].unique()
print(f'target 상품 수: {len(target_products):,}')

# prior 구매에서 target 상품 + user_id 확보
pp_co = prior_products[prior_products['product_id'].isin(target_products)].copy()
pp_co = pp_co.merge(prior_orders[['order_id','user_id']], on='order_id', how='inner')

# 바구니 크기 2~30으로 제한 (너무 큰 바구니는 노이즈 / 조합 폭발 방지)
basket_size = pp_co.groupby('order_id')['product_id'].transform('count')
pp_co = pp_co[basket_size.between(2, 30)].copy()

print(f'사용 주문 수   : {pp_co["order_id"].nunique():,}')
print(f'사용 구매 기록 : {len(pp_co):,}')

# 상품/주문 인덱싱
pid2idx = {p: i for i, p in enumerate(target_products)}
idx2pid = {i: p for p, i in pid2idx.items()}
order_ids_co = pp_co['order_id'].unique()
oid2idx = {o: i for i, o in enumerate(order_ids_co)}

row_idx = pp_co['order_id'].map(oid2idx).values
col_idx = pp_co['product_id'].map(pid2idx).values
vals    = np.ones(len(pp_co), dtype=np.float32)

mat = csr_matrix(
    (vals, (row_idx, col_idx)),
    shape=(len(order_ids_co), len(target_products))
)
print(f'\nsparse 행렬: {mat.shape[0]:,} 주문 × {mat.shape[1]:,} 상품')
print(f'비영 원소  : {mat.nnz:,}')

In [ ]:
# 배치 방식으로 A^T @ A 계산 → 상품별 top-K 공동구매 파트너 추출
# batch.T @ mat → (BATCH, n_products): batch 내 각 상품과 전체 상품 간 공동구매 횟수
K_TOP = 10
BATCH = 200
n_products = len(target_products)

copurchase_records = []  # (product_id, partner_id, co_count, rank)

print(f'공동구매 계산 중... (총 {n_products:,}개 상품, 배치={BATCH}, K={K_TOP})')
for start in range(0, n_products, BATCH):
    end   = min(start + BATCH, n_products)
    batch = mat[:, start:end]             # (n_orders, BATCH)
    co    = batch.T.dot(mat).toarray()   # (BATCH, n_products) — dense 변환

    for local_i in range(end - start):
        global_i = start + local_i
        co[local_i, global_i] = 0        # 자기 자신 제거
        top_idx = np.argsort(co[local_i])[::-1][:K_TOP]
        for rank, j in enumerate(top_idx):
            cnt = int(co[local_i, j])
            if cnt > 0:
                copurchase_records.append(
                    (idx2pid[global_i], idx2pid[j], cnt, rank + 1)
                )

    if start % (BATCH * 20) == 0 and start > 0:
        print(f'  {end:,}/{n_products:,} 완료')

copurchase_pairs = pd.DataFrame(
    copurchase_records,
    columns=['product_id','partner_id','co_count','co_rank']
)

# 상품별 공동구매 총합으로 정규화 → 상대적 중요도 (0~1)
copurchase_pairs['co_score_norm'] = (
    copurchase_pairs['co_count'] /
    copurchase_pairs.groupby('product_id')['co_count'].transform('sum').clip(1)
).astype(np.float32)

print(f'\ncopurchase_pairs 행 수: {len(copurchase_pairs):,}')
print(f'공동구매 파트너가 있는 상품 수: {copurchase_pairs["product_id"].nunique():,}')
copurchase_pairs.head(10)

In [ ]:
# 유저 구매 히스토리: prior 전체 기준 (unique user-product 쌍)
user_hist = (
    prior_products
    .merge(prior_orders[['order_id','user_id']], on='order_id', how='inner')
    [['user_id','product_id']]
    .drop_duplicates()
    .rename(columns={'product_id': 'partner_id'})  # join 키 이름 맞춤
)
user_hist['in_history'] = np.int8(1)

print(f'user_hist 행 수         : {len(user_hist):,}')
print(f'유저 수                 : {user_hist["user_id"].nunique():,}')
print(f'유저당 평균 구매 상품 수: {len(user_hist) / user_hist["user_id"].nunique():.1f}')

In [ ]:
# (user, product)별 공동구매 친화도 계산 — 청크 방식 (메모리 절약)
#
# [계산 원리]
# 1) target × copurchase_pairs → product의 top-K 파트너 붙이기 (1행 → K행으로 확장)
# 2) 확장된 df × user_hist → 각 파트너를 유저가 산 적 있는지 확인
# 3) 집계: (user, product) 단위로 점수 합산

CHUNK = 500_000
total = len(target)
results = []

print(f'공동구매 친화도 계산 중... (총 {total:,}행, 청크={CHUNK:,})')
for start in range(0, total, CHUNK):
    chunk = target[['user_id','product_id']].iloc[start:start + CHUNK].copy()

    # product_id 기준으로 co-purchase 파트너 붙이기
    exp = chunk.merge(
        copurchase_pairs[['product_id','partner_id','co_count','co_score_norm']],
        on='product_id', how='left'
    )

    # 유저 히스토리에 있는지 확인
    exp = exp.merge(user_hist, on=['user_id','partner_id'], how='left')
    exp['in_history']  = exp['in_history'].fillna(0)
    exp['weighted_hit'] = exp['co_score_norm'].fillna(0) * exp['in_history']

    # (user, product) 단위로 집계
    agg = (
        exp.groupby(['user_id','product_id'], sort=False)
        .agg(
            copurchase_match_count  =('in_history', 'sum'),
            copurchase_partner_k    =('partner_id', 'count'),
            copurchase_weighted_score=('weighted_hit', 'sum')
        )
        .reset_index()
    )
    results.append(agg)
    print(f'  {min(start + CHUNK, total):,}/{total:,} 완료')

copurchase_final = pd.concat(results, ignore_index=True)

# 최종 비율 피처: 매칭 수 / 파트너 수
copurchase_final['copurchase_ratio'] = (
    copurchase_final['copurchase_match_count'] /
    copurchase_final['copurchase_partner_k'].clip(1)
).astype(np.float32)
copurchase_final['copurchase_weighted_score'] = (
    copurchase_final['copurchase_weighted_score'].astype(np.float32)
)
copurchase_final['copurchase_match_count'] = (
    copurchase_final['copurchase_match_count'].astype(np.int8)
)

COPURCHASE_COLS = [
    'user_id','product_id',
    'copurchase_ratio','copurchase_weighted_score','copurchase_match_count'
]
copurchase_final = copurchase_final[COPURCHASE_COLS].copy()

print(f'\ncopurchase_final 행 수: {len(copurchase_final):,}')
print(f'copurchase_ratio > 0 비율: {(copurchase_final["copurchase_ratio"] > 0).mean():.3f}')
copurchase_final.describe().T.round(4)

---
## 2. 피처 검증 — 새 피처와 label의 관계 확인

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# label과 함께 병합해서 관계 확인
label_df = pd.read_csv(PREP / 'k-pick_total_v3.csv', usecols=['user_id','product_id','label'])

check = (
    label_df
    .merge(timing_final,     on=['user_id','product_id'], how='left')
    .merge(copurchase_final, on=['user_id','product_id'], how='left')
)

new_features = [
    'is_overdue','timing_ratio','days_until_expected',
    'up_purchase_regularity','up_avg_purchase_interval',
    'copurchase_ratio','copurchase_weighted_score','copurchase_match_count'
]

print('=== 재구매(label=1) vs 미구매(label=0) — 새 피처 평균값 비교 ===')
print(check.groupby('label')[new_features].mean().round(4).T.to_string())

In [ ]:
# is_overdue(구매 주기 초과)와 재구매율 관계
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) is_overdue별 재구매율
overdue_reorder = check.groupby('is_overdue')['label'].mean()
axes[0].bar(['아직 안 됨(0)', '주기 초과(1)'], overdue_reorder.values, color=['steelblue','tomato'])
axes[0].set_title('구매 주기 초과 여부별 재구매율')
axes[0].set_ylabel('재구매율')
for i, v in enumerate(overdue_reorder.values):
    axes[0].text(i, v + 0.002, f'{v:.3f}', ha='center')

# 2) timing_ratio 구간별 재구매율
check['timing_bin'] = pd.cut(check['timing_ratio'].clip(0, 3),
                              bins=[0, 0.5, 1.0, 1.5, 2.0, 3.0],
                              labels=['0~0.5', '0.5~1.0', '1.0~1.5', '1.5~2.0', '2.0~3.0'])
timing_reorder = check.groupby('timing_bin', observed=True)['label'].mean()
axes[1].bar(timing_reorder.index.astype(str), timing_reorder.values, color='mediumseagreen')
axes[1].set_title('timing_ratio 구간별 재구매율')
axes[1].set_xlabel('timing_ratio 구간 (경과일 / 평균주기)')
axes[1].set_ylabel('재구매율')

# 3) copurchase_match_count별 재구매율
cp_reorder = check.groupby('copurchase_match_count')['label'].mean().head(8)
axes[2].bar(cp_reorder.index.astype(str), cp_reorder.values, color='mediumpurple')
axes[2].set_title('공동구매 매칭 수별 재구매율')
axes[2].set_xlabel('공동구매 파트너 매칭 수')
axes[2].set_ylabel('재구매율')

plt.tight_layout()
plt.show()

---
## 3. k-pick_total_v3와 병합 → v4 저장

In [ ]:
print('k-pick_total_v3.csv 로드 중...')
v3 = pd.read_csv(PREP / 'k-pick_total_v3.csv')
print(f'v3: {v3.shape}')

v4 = (
    v3
    .merge(timing_final,     on=['user_id','product_id'], how='left')
    .merge(copurchase_final, on=['user_id','product_id'], how='left')
)

# 결측치 처리
v4['up_interval_count']         = v4['up_interval_count'].fillna(0).astype(np.int16)
v4['is_overdue']                = v4['is_overdue'].fillna(0).astype(np.int8)
v4['timing_ratio']              = v4['timing_ratio'].fillna(1.0)   # 기본: 정확히 주기 도달
v4['copurchase_match_count']    = v4['copurchase_match_count'].fillna(0).astype(np.int8)
v4['copurchase_ratio']          = v4['copurchase_ratio'].fillna(0).astype(np.float32)
v4['copurchase_weighted_score'] = v4['copurchase_weighted_score'].fillna(0).astype(np.float32)

for col in ['up_avg_purchase_interval','up_std_purchase_interval',
            'up_purchase_regularity','days_until_expected']:
    v4[col] = v4[col].fillna(v4[col].median()).astype(np.float32)

# 결과 확인
new_cols = [c for c in v4.columns if c not in v3.columns]
print(f'\n추가된 피처 ({len(new_cols)}개):')
for c in new_cols:
    print(f'  {c}')

print(f'\nv3 컬럼: {v3.shape[1]}  →  v4 컬럼: {v4.shape[1]}')
missing = v4[new_cols].isnull().sum()
print(f'결측치: {missing[missing > 0].to_dict() or "없음"}')

In [ ]:
out_path = PREP / 'k-pick_total_v4.csv'
v4.to_csv(out_path, index=False, float_format='%.4f')

import os
size_mb = os.path.getsize(out_path) / 1024 / 1024
print(f'저장 완료: {out_path}')
print(f'행: {len(v4):,}  |  컬럼: {v4.shape[1]}  |  크기: {size_mb:.1f} MB')